# ECG heartbeat classification (1D CNN)
Author: Abdullah Ghani, ECE Paris (Data & AI)

Five-class single-beat classification (N normal, S supraventricular, V ventricular, F fusion, Q unknown) on the public Kaggle dataset `shayanfazeli/heartbeat` (preprocessed MIT-BIH arrhythmia beats, 187 samples at 125 Hz). The dataset's own train/test files are used; 10% of train is held out for validation and model selection.

> Research prototype. Not a medical device. Not for clinical use. Runtime: Colab GPU (T4).

In [ ]:
!pip install -q kagglehub
import os, glob, json, random, numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
SEED=42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
import kagglehub
root=kagglehub.dataset_download("shayanfazeli/heartbeat")
print(glob.glob(root+"/*"))
tr=pd.read_csv(glob.glob(root+"/**/mitbih_train.csv",recursive=True)[0],header=None)
te=pd.read_csv(glob.glob(root+"/**/mitbih_test.csv",recursive=True)[0],header=None)
CL=["N","S","V","F","Q"]
Xtr=tr.iloc[:,:-1].values.astype("float32")[...,None]; ytr=tr.iloc[:,-1].values.astype(int)
Xte=te.iloc[:,:-1].values.astype("float32")[...,None]; yte=te.iloc[:,-1].values.astype(int)
print(Xtr.shape, np.bincount(ytr), Xte.shape, np.bincount(yte))

In [ ]:
Xt,Xv,yt,yv=train_test_split(Xtr,ytr,test_size=0.1,stratify=ytr,random_state=SEED)
def block(x,f,k):
    x=layers.Conv1D(f,k,padding="same")(x); x=layers.BatchNormalization()(x); x=layers.ReLU()(x); return x
inp=layers.Input((187,1)); x=block(inp,32,7); x=layers.MaxPool1D(2)(x); x=block(x,64,5); x=layers.MaxPool1D(2)(x); x=block(x,128,3); x=layers.MaxPool1D(2)(x)
x=layers.GlobalAveragePooling1D()(x); x=layers.Dropout(0.3)(x); out=layers.Dense(5,activation="softmax")(x)
m=models.Model(inp,out); m.compile(optimizers.Adam(1e-3),"sparse_categorical_crossentropy",metrics=["accuracy"])
cnt=np.bincount(yt); cw={i:float((len(yt)/(5*cnt[i]))**0.5) for i in range(5)}   # softened class weights
h=m.fit(Xt,yt,validation_data=(Xv,yv),epochs=30,batch_size=256,class_weight=cw,verbose=2,
        callbacks=[callbacks.EarlyStopping(patience=5,restore_best_weights=True,monitor="val_loss")])

In [ ]:
p=m.predict(Xte,batch_size=512,verbose=0); pred=p.argmax(1)
rep=classification_report(yte,pred,target_names=CL,output_dict=True,digits=4); print(classification_report(yte,pred,target_names=CL,digits=4))
aucs={c:float(roc_auc_score((yte==i).astype(int),p[:,i])) for i,c in enumerate(CL)}
res={"test_beats":int(len(yte)),"accuracy":float((pred==yte).mean()),"macro_f1":float(f1_score(yte,pred,average="macro")),"macro_auc":float(np.mean(list(aucs.values()))),
 "recall":{c:float(rep[c]["recall"]) for c in CL},"precision":{c:float(rep[c]["precision"]) for c in CL},"auc":aucs,"confusion_matrix":confusion_matrix(yte,pred).tolist(),"classes":CL}
json.dump(res,open("metrics.json","w"),indent=2); print(json.dumps(res,indent=2))
cm=confusion_matrix(yte,pred,normalize="true"); fig,ax=plt.subplots(figsize=(5,4.5)); im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1)
for i in range(5):
    for j in range(5): ax.text(j,i,f"{cm[i,j]:.2f}",ha="center",va="center",fontsize=8)
ax.set_xticks(range(5));ax.set_yticks(range(5));ax.set_xticklabels(CL);ax.set_yticklabels(CL);ax.set_xlabel("predicted");ax.set_ylabel("true");ax.set_title("Test confusion matrix (row-normalised)")
plt.tight_layout(); plt.savefig("confusion_matrix.png",dpi=150); plt.show()
fig,ax=plt.subplots(1,5,figsize=(14,2.4))
for i,a in enumerate(ax): a.plot(Xte[np.where(yte==i)[0][0],:,0]); a.set_title(CL[i]); a.set_xticks([])
plt.tight_layout(); plt.savefig("example_beats.png",dpi=130); plt.show()
plt.figure(figsize=(5,3)); plt.plot(h.history["loss"],label="train"); plt.plot(h.history["val_loss"],label="val"); plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss"); plt.tight_layout(); plt.savefig("learning_curve.png",dpi=130); plt.show()
m.save("ecg_cnn.keras")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dst="/content/drive/MyDrive/Portfolio imaging projects/ecg-arrhythmia"; os.makedirs(dst,exist_ok=True)
import shutil
for f in ["ecg_cnn.keras","metrics.json","confusion_matrix.png","example_beats.png","learning_curve.png"]: shutil.copy(f,dst)
print(os.listdir(dst))

## Limitations
- The Kaggle beat files mix beats from the same recordings across train and test, so this is not an inter-patient evaluation and the scores are optimistic compared with patient-independent protocols such as the AAMI/de Chazal split.
- Single-beat classification of preprocessed 187-sample segments; no rhythm context, no detection of beats from raw ECG.
- Class imbalance (N is about 83% of beats); read the per-class recall, not just accuracy.
- Two source databases (MIT-BIH arrhythmia), small number of subjects, no external validation.